# Grover's Algorithm — 10-Qubit Search (Qiskit)

## Objective
Implement Grover's search algorithm from scratch using Qiskit to locate **two marked states**
in a 10-qubit search space (2¹⁰ = 1024 possible states).

**Marked states:**
- `0110011010`
- `1101010001`

### Components built manually:
1. Superposition state (Hadamard layer)
2. Oracle — phase-flips the two marked states
3. Diffusion operator — amplifies marked state probabilities
4. Grover iterations — repeated oracle + diffusion
5. Measurement + histogram

> ⚠️ No built-in Qiskit Grover implementations are used. Everything is built using basic quantum gates.

In [1]:
# ─────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — prevents blank figures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

---

## Step 1 — Problem Setup

We define our 10-qubit search space and specify the two target (marked) states.

The search space has **2¹⁰ = 1024** possible bit strings.  
Grover's algorithm needs approximately **π/4 × √(N/M)** iterations to find M marked items
among N total items.

For N = 1024, M = 2:
> Optimal iterations ≈ π/4 × √(1024/2) ≈ **18 iterations**

We will test with 1, 3, 5, and 10 iterations to observe how probability builds up.

In [2]:
# ─────────────────────────────────────────────
# Problem Setup
# ─────────────────────────────────────────────

n_qubits = 10  # 10-qubit search space → 2^10 = 1024 states

# Marked states — changing these automatically updates the oracle
targets = ["0110011010", "1101010001"]

N = 2 ** n_qubits   # Total states = 1024
M = len(targets)    # Number of marked states = 2

# Optimal number of Grover iterations
optimal_iters = int(np.round((np.pi / 4) * np.sqrt(N / M)))
print(f"Search space size      : {N}")
print(f"Number of marked states: {M}")
print(f"Optimal Grover iters   : {optimal_iters}")
print(f"Marked states          : {targets}")

Search space size      : 1024
Number of marked states: 2
Optimal Grover iters   : 18
Marked states          : ['0110011010', '1101010001']


---

## Stage 1 Circuit — Superposition (Hadamard Layer)

The first stage applies **Hadamard gates to all 10 qubits**, creating an equal superposition
of all 1024 basis states.

Each qubit starts in |0⟩ and after H becomes:

> |0⟩  →  (|0⟩ + |1⟩) / √2

After applying H to all 10 qubits:

> |0⟩⊗¹⁰  →  (1/√1024) × Σ |x⟩  for all x ∈ {0,1}¹⁰

The circuit below shows only this first stage.

In [3]:
# ─────────────────────────────────────────────
# STAGE 1 CIRCUIT: Superposition (Hadamard Layer)
# ─────────────────────────────────────────────
stage1 = QuantumCircuit(n_qubits, name="Stage 1: Superposition")
stage1.h(range(n_qubits))

print("=" * 50)
print("  STAGE 1 — Superposition (Hadamard Layer)")
print("=" * 50)
print(stage1.draw('text'))

# ── FIX: save BEFORE show, close after ───────
fig1 = stage1.draw('mpl', style='clifford')
fig1.set_size_inches(12, 4)                          # explicit size
fig1.suptitle("Stage 1: Hadamard Layer — Equal Superposition of All 1024 States",
              fontsize=11, fontweight='bold')
fig1.savefig("stage1_superposition.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig1)
print("Saved: stage1_superposition.png")


  STAGE 1 — Superposition (Hadamard Layer)
     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     ├───┤
q_3: ┤ H ├
     ├───┤
q_4: ┤ H ├
     ├───┤
q_5: ┤ H ├
     ├───┤
q_6: ┤ H ├
     ├───┤
q_7: ┤ H ├
     ├───┤
q_8: ┤ H ├
     ├───┤
q_9: ┤ H ├
     └───┘
Saved: stage1_superposition.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\1679099744.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## Stage 2 Circuit — Oracle Construction

The **oracle** is the core of Grover's algorithm. It:
- Applies a **phase flip of −1** to each marked state
- Leaves all other states **unchanged**

### How it works (for each marked state):
For a marked state like `0110011010`, we want to flip its phase.
We do this using a **multi-controlled Z gate (MCZ)**:

| Step | Operation | Purpose |
|------|-----------|---------|
| 1 | X gate on qubits where target bit = `0` | Convert `\|0⟩` → `\|1⟩` so MCZ fires correctly |
| 2 | Multi-controlled Z (MCX + H sandwich) | Phase-flip `\|111...1⟩` which maps to the target |
| 3 | X gate again (uncompute) | Restore qubits to original state |

This is repeated for **both** marked states, so both get phase-flipped.

The circuit below shows the oracle acting on a **4-qubit example** for readability,
followed by the full 10-qubit oracle structure.

In [4]:
# ─────────────────────────────────────────────
# STAGE 2 CIRCUIT: Oracle
# ─────────────────────────────────────────────
def build_oracle(n, targets):
    oracle = QuantumCircuit(n, name="Oracle")
    for target in targets:
        for i, bit in enumerate(reversed(target)):
            if bit == '0':
                oracle.x(i)
        oracle.h(n - 1)
        oracle.mcx(list(range(n - 1)), n - 1)
        oracle.h(n - 1)
        for i, bit in enumerate(reversed(target)):
            if bit == '0':
                oracle.x(i)
        oracle.barrier()
    return oracle

# ── 4-qubit example ───────────────────────────
print("=" * 60)
print("  STAGE 2 — Oracle (4-qubit example)")
print("=" * 60)
example_targets = ["0110", "1001"]
oracle_4bit = build_oracle(4, example_targets)
print(oracle_4bit.draw('text'))

fig2a = oracle_4bit.draw('mpl', style='clifford')
fig2a.set_size_inches(10, 4)
fig2a.suptitle("Stage 2: Oracle — 4-Qubit Example\nPhase-flips: 0110 and 1001",
               fontsize=11, fontweight='bold')
fig2a.savefig("stage2_oracle_4bit.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig2a)

# ── Full 10-qubit oracle ──────────────────────
print("\n" + "=" * 60)
print("  STAGE 2 — Oracle (full 10-qubit)")
print("=" * 60)
oracle_10bit = build_oracle(n_qubits, targets)
print(oracle_10bit.decompose().draw('text', fold=120))

fig2b = oracle_10bit.draw('mpl', style='clifford', fold=30)
fig2b.set_size_inches(14, 5)
fig2b.suptitle("Stage 2: Oracle — Full 10-Qubit\nPhase-flips: 0110011010 and 1101010001",
               fontsize=11, fontweight='bold')
fig2b.savefig("stage2_oracle_10bit.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig2b)
print(f"Oracle gate count (10-qubit): {oracle_10bit.size()} gates")
print("Saved: stage2_oracle_4bit.png, stage2_oracle_10bit.png")


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\4069626988.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  STAGE 2 — Oracle (4-qubit example)
     ┌───┐          ┌───┐      ░                 ░ 
q_0: ┤ X ├───────■──┤ X ├──────░────────■────────░─
     └───┘       │  └───┘      ░ ┌───┐  │  ┌───┐ ░ 
q_1: ────────────■─────────────░─┤ X ├──■──┤ X ├─░─
                 │             ░ ├───┤  │  ├───┤ ░ 
q_2: ────────────■─────────────░─┤ X ├──■──┤ X ├─░─
     ┌───┐┌───┐┌─┴─┐┌───┐┌───┐ ░ ├───┤┌─┴─┐├───┤ ░ 
q_3: ┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├─░─┤ H ├┤ X ├┤ H ├─░─
     └───┘└───┘└───┘└───┘└───┘ ░ └───┘└───┘└───┘ ░ 

  STAGE 2 — Oracle (full 10-qubit)
     ┌──────────┐                          ┌──────────┐                           ░                           »
q_0: ┤ U(π,0,π) ├────────────────────■─────┤ U(π,0,π) ├───────────────────────────░─────────────────────■─────»
     └──────────┘                    │     └──────────┘                           ░  ┌──────────┐       │     »
q_1: ────────────────────────────────■────────────────────────────────────────────░──┤ U(π,0,π) ├───────■─────»
     ┌─────

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\4069626988.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## Stage 3 Circuit — Diffusion Operator

The **diffusion operator** (Grover diffusion / inversion-about-the-mean) amplifies the
probability of marked states by reflecting all amplitudes about their average.

### Structure: H → U₀ → H

| Step | Operation | Purpose |
|------|-----------|---------|
| H⊗ⁿ | Hadamard on all qubits | Rotate to the \|0...0⟩ basis |
| U₀  | Phase flip all states **except** \|0...0⟩ | Invert amplitudes about zero |
| H⊗ⁿ | Hadamard on all qubits | Rotate back to computational basis |

### U₀ implementation:
U₀ flips the sign of every state **except** |0...0⟩.  
Implementation: X on all qubits → MCZ → X on all qubits.

The circuit below shows the diffusion operator for a **4-qubit example** for readability,
followed by the full 10-qubit version.

In [5]:
# ─────────────────────────────────────────────
# STAGE 3 CIRCUIT: Diffusion Operator
# ─────────────────────────────────────────────
def build_diffusion(n):
    diffusion = QuantumCircuit(n, name="Diffusion")
    diffusion.h(range(n))
    diffusion.x(range(n))
    diffusion.h(n - 1)
    diffusion.mcx(list(range(n - 1)), n - 1)
    diffusion.h(n - 1)
    diffusion.x(range(n))
    diffusion.h(range(n))
    return diffusion

# ── 4-qubit diffusion ─────────────────────────
print("=" * 60)
print("  STAGE 3 — Diffusion Operator (4-qubit example)")
print("=" * 60)
diff_4bit = build_diffusion(4)
print(diff_4bit.draw('text'))

fig3a = diff_4bit.draw('mpl', style='clifford')
fig3a.set_size_inches(10, 4)
fig3a.suptitle("Stage 3: Diffusion Operator — 4-Qubit Example\nH → U₀ → H",
               fontsize=11, fontweight='bold')
fig3a.savefig("stage3_diffusion_4bit.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig3a)

# ── Full 10-qubit diffusion ───────────────────
print("\n" + "=" * 60)
print("  STAGE 3 — Diffusion Operator (full 10-qubit)")
print("=" * 60)
diff_10bit = build_diffusion(n_qubits)
print(diff_10bit.decompose().draw('text', fold=120))

fig3b = diff_10bit.draw('mpl', style='clifford', fold=30)
fig3b.set_size_inches(14, 5)
fig3b.suptitle("Stage 3: Diffusion Operator — Full 10-Qubit\nH → X⊗¹⁰ → MCZ → X⊗¹⁰ → H",
               fontsize=11, fontweight='bold')
fig3b.savefig("stage3_diffusion_10bit.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig3b)
print(f"Diffusion gate count (10-qubit): {diff_10bit.size()} gates")
print("Saved: stage3_diffusion_4bit.png, stage3_diffusion_10bit.png")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\960821051.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  STAGE 3 — Diffusion Operator (4-qubit example)
     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_2: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_3: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘

  STAGE 3 — Diffusion Operator (full 10-qubit)
     ┌────────────┐┌──────────┐                          ┌──────────┐┌────────────┐                          
q_0: ┤ U(π/2,0,π) ├┤ U(π,0,π) ├────────────────────■─────┤ U(π,0,π) ├┤ U(π/2,0,π) ├──────────────────────────
     ├────────────┤├──────────┤                    │     ├──────────┤├────────────┤                          
q_1: ┤ U(π/2,0,π) ├┤ U(π,0,π) ├────────────────────■─────┤ U(π,0,π) ├┤ U(π/2,0,π) ├──────────────────────────
     ├────────────┤├──────────┤                    │     ├──────────┤├────────────┤           

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\960821051.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## Stage 4 Circuit — One Full Grover Iteration

A single **Grover iteration** = Oracle + Diffusion applied once after the initial superposition.
```
|0⟩⊗¹⁰ ──[ H⊗¹⁰ ]──[ Oracle ]──[ Diffusion ]──
```

The circuit below shows **1 complete Grover iteration** (no measurement yet)
so you can see all three stages together in sequence.

In [6]:
# ─────────────────────────────────────────────
# Helper — strip barriers before converting to gate
# ─────────────────────────────────────────────
def circuit_to_gate_clean(qc, label=None):
    """Convert a QuantumCircuit to a Gate, stripping barriers first."""
    clean = QuantumCircuit(qc.num_qubits)
    for instruction in qc.data:
        if instruction.operation.name != "barrier":
            clean.append(instruction)
    return clean.to_gate(label=label)

In [7]:
# ─────────────────────────────────────────────
# STAGE 4 CIRCUIT: One Full Grover Iteration
# ─────────────────────────────────────────────
def build_grover_circuit(n, targets, num_iterations):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    qc.barrier(label="Superposition")

    oracle_gate    = circuit_to_gate_clean(build_oracle(n, targets),    label="Oracle")
    diffusion_gate = circuit_to_gate_clean(build_diffusion(n),          label="Diffusion")

    for i in range(num_iterations):
        qc.append(oracle_gate,    range(n))
        qc.barrier(label=f"Oracle {i+1}")
        qc.append(diffusion_gate, range(n))
        qc.barrier(label=f"Diffusion {i+1}")

    qc.measure(range(n), range(n))
    return qc

# ── Structure view ────────────────────────────
print("=" * 60)
print("  STAGE 4 — One Grover Iteration (structure view)")
print("=" * 60)

qc_1iter_view = QuantumCircuit(n_qubits)
qc_1iter_view.h(range(n_qubits))
qc_1iter_view.barrier()
qc_1iter_view.append(
    circuit_to_gate_clean(build_oracle(n_qubits, targets), label="Oracle"),
    range(n_qubits)
)
qc_1iter_view.barrier()
qc_1iter_view.append(
    circuit_to_gate_clean(build_diffusion(n_qubits), label="Diffusion"),
    range(n_qubits)
)
qc_1iter_view.barrier()
print(qc_1iter_view.draw('text'))

fig4 = qc_1iter_view.draw('mpl', style='clifford', fold=40)
fig4.set_size_inches(14, 5)
fig4.suptitle("Stage 4: One Complete Grover Iteration\nH⊗¹⁰ → Oracle → Diffusion",
              fontsize=11, fontweight='bold')
fig4.savefig("stage4_one_iteration.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig4)
print("Saved: stage4_one_iteration.png")

  STAGE 4 — One Grover Iteration (structure view)
     ┌───┐ ░ ┌─────────┐ ░ ┌────────────┐ ░ 
q_0: ┤ H ├─░─┤0        ├─░─┤0           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_1: ┤ H ├─░─┤1        ├─░─┤1           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_2: ┤ H ├─░─┤2        ├─░─┤2           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_3: ┤ H ├─░─┤3        ├─░─┤3           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_4: ┤ H ├─░─┤4        ├─░─┤4           ├─░─
     ├───┤ ░ │  Oracle │ ░ │  Diffusion │ ░ 
q_5: ┤ H ├─░─┤5        ├─░─┤5           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_6: ┤ H ├─░─┤6        ├─░─┤6           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_7: ┤ H ├─░─┤7        ├─░─┤7           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_8: ┤ H ├─░─┤8        ├─░─┤8           ├─░─
     ├───┤ ░ │         │ ░ │            │ ░ 
q_9: ┤ H ├─░─┤9        ├─░─┤9           ├─░─
     └───┘ ░ └─────────┘ ░ └────────────┘ ░ 
Saved

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\3564042411.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## Stage 5 Circuit — Full Grover Circuit with Measurement (k iterations)

Now we show the **complete final circuit** including measurement, for each iteration count.
```
|0⟩⊗¹⁰ → H⊗¹⁰ → [Oracle → Diffusion]×k → Measure → Classical bits
```

The circuit is shown as **named gate boxes** (Oracle, Diffusion) for clarity.
Below we display the 3-iteration version as a representative example.

In [8]:
# ─────────────────────────────────────────────
# STAGE 5 CIRCUIT: Full Grover Circuit with Measurement
# ─────────────────────────────────────────────
oracle_gate    = circuit_to_gate_clean(build_oracle(n_qubits, targets), label="Oracle")
diffusion_gate = circuit_to_gate_clean(build_diffusion(n_qubits),       label="Diffusion")

for k in [1, 3, 5, 10]:
    qc_view = QuantumCircuit(n_qubits, n_qubits)
    qc_view.h(range(n_qubits))
    qc_view.barrier()
    for i in range(k):
        qc_view.append(oracle_gate,    range(n_qubits))
        qc_view.barrier()
        qc_view.append(diffusion_gate, range(n_qubits))
        qc_view.barrier()
    qc_view.measure(range(n_qubits), range(n_qubits))

    print(f"\n{'='*60}")
    print(f"  FULL CIRCUIT — {k} Grover Iteration(s) + Measurement")
    print(f"{'='*60}")
    print(qc_view.draw('text'))

    fig = qc_view.draw('mpl', style='clifford', fold=40)
    fig.set_size_inches(14, 5)
    fig.suptitle(f"Full Grover Circuit — {k} Iteration(s)\n"
                 f"H⊗¹⁰ → [Oracle → Diffusion]×{k} → Measure",
                 fontsize=11, fontweight='bold')
    fname = f"stage5_full_circuit_{k}iter.png"
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Saved: {fname}")
    print(f"Total gates: {qc_view.decompose().size()}")



  FULL CIRCUIT — 1 Grover Iteration(s) + Measurement
      ┌───┐ ░ ┌─────────┐ ░ ┌────────────┐ ░ ┌─┐                           
 q_0: ┤ H ├─░─┤0        ├─░─┤0           ├─░─┤M├───────────────────────────
      ├───┤ ░ │         │ ░ │            │ ░ └╥┘┌─┐                        
 q_1: ┤ H ├─░─┤1        ├─░─┤1           ├─░──╫─┤M├────────────────────────
      ├───┤ ░ │         │ ░ │            │ ░  ║ └╥┘┌─┐                     
 q_2: ┤ H ├─░─┤2        ├─░─┤2           ├─░──╫──╫─┤M├─────────────────────
      ├───┤ ░ │         │ ░ │            │ ░  ║  ║ └╥┘┌─┐                  
 q_3: ┤ H ├─░─┤3        ├─░─┤3           ├─░──╫──╫──╫─┤M├──────────────────
      ├───┤ ░ │         │ ░ │            │ ░  ║  ║  ║ └╥┘┌─┐               
 q_4: ┤ H ├─░─┤4        ├─░─┤4           ├─░──╫──╫──╫──╫─┤M├───────────────
      ├───┤ ░ │  Oracle │ ░ │  Diffusion │ ░  ║  ║  ║  ║ └╥┘┌─┐            
 q_5: ┤ H ├─░─┤5        ├─░─┤5           ├─░──╫──╫──╫──╫──╫─┤M├────────────
      ├───┤ ░ │         │ ░ │     

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\1544441777.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: stage5_full_circuit_1iter.png
Total gates: 89

  FULL CIRCUIT — 3 Grover Iteration(s) + Measurement
      ┌───┐ ░ ┌─────────┐ ░ ┌────────────┐ ░ ┌─────────┐ ░ ┌────────────┐ ░ »
 q_0: ┤ H ├─░─┤0        ├─░─┤0           ├─░─┤0        ├─░─┤0           ├─░─»
      ├───┤ ░ │         │ ░ │            │ ░ │         │ ░ │            │ ░ »
 q_1: ┤ H ├─░─┤1        ├─░─┤1           ├─░─┤1        ├─░─┤1           ├─░─»
      ├───┤ ░ │         │ ░ │            │ ░ │         │ ░ │            │ ░ »
 q_2: ┤ H ├─░─┤2        ├─░─┤2           ├─░─┤2        ├─░─┤2           ├─░─»
      ├───┤ ░ │         │ ░ │            │ ░ │         │ ░ │            │ ░ »
 q_3: ┤ H ├─░─┤3        ├─░─┤3           ├─░─┤3        ├─░─┤3           ├─░─»
      ├───┤ ░ │         │ ░ │            │ ░ │         │ ░ │            │ ░ »
 q_4: ┤ H ├─░─┤4        ├─░─┤4           ├─░─┤4        ├─░─┤4           ├─░─»
      ├───┤ ░ │  Oracle │ ░ │  Diffusion │ ░ │  Oracle │ ░ │  Diffusion │ ░ »
 q_5: ┤ H ├─░─┤5        ├─░─┤5     

---

## Step 5 — Execute and Observe Results

We now run the Grover circuit on the **Qiskit AerSimulator** with **1024 shots** for each
iteration count and plot the measurement histograms.

**What to look for:**
- At **1 iteration**: marked states start to emerge but probabilities are still low
- At **3–5 iterations**: marked states become clearly dominant
- At **10 iterations**: probability may start to decrease slightly (over-rotation)
- At **~18 iterations**: peak probability (theoretical optimum)

In [9]:
# ─────────────────────────────────────────────
# Execute Grover Circuit — Multiple Iteration Counts
# ─────────────────────────────────────────────
simulator   = AerSimulator()
shots       = 1024
iter_counts = [1, 3, 5, 10]
all_counts  = {}

for k in iter_counts:
    print(f"\n--- Running Grover with {k} iteration(s) ---")
    qc       = build_grover_circuit(n_qubits, targets, k)
    compiled = transpile(qc, simulator)
    job      = simulator.run(compiled, shots=shots)
    result   = job.result()
    counts   = result.get_counts()
    all_counts[k] = counts

    top5 = sorted(counts.items(), key=lambda x: -x[1])[:5]
    print(f"  Top 5 outcomes:")
    for state, count in top5:
        prob   = count / shots * 100
        marker = " ← MARKED ✓" if state in targets else ""
        print(f"    {state} : {count:4d} shots  ({prob:.1f}%){marker}")



--- Running Grover with 1 iteration(s) ---
  Top 5 outcomes:
    0110011010 :   14 shots  (1.4%) ← MARKED ✓
    1101010001 :   14 shots  (1.4%) ← MARKED ✓
    1000001101 :    5 shots  (0.5%)
    1010110000 :    5 shots  (0.5%)
    0011011101 :    5 shots  (0.5%)

--- Running Grover with 3 iteration(s) ---
  Top 5 outcomes:
    0110011010 :   42 shots  (4.1%) ← MARKED ✓
    1101010001 :   40 shots  (3.9%) ← MARKED ✓
    0000010110 :    5 shots  (0.5%)
    1000111100 :    4 shots  (0.4%)
    1110010100 :    4 shots  (0.4%)

--- Running Grover with 5 iteration(s) ---
  Top 5 outcomes:
    0110011010 :  119 shots  (11.6%) ← MARKED ✓
    1101010001 :  106 shots  (10.4%) ← MARKED ✓
    0101111010 :    5 shots  (0.5%)
    1100011001 :    5 shots  (0.5%)
    0001110010 :    5 shots  (0.5%)

--- Running Grover with 10 iteration(s) ---
  Top 5 outcomes:
    0110011010 :  362 shots  (35.4%) ← MARKED ✓
    1101010001 :  304 shots  (29.7%) ← MARKED ✓
    0010100101 :    3 shots  (0.3%)
    1101001

---

## Results — Measurement Histograms

The histograms below show the full probability distribution for each iteration count.  
**Crimson bars** = marked states (`0110011010` and `1101010001`).  
**Steel blue bars** = all other states.

The marked states should clearly dominate as iterations increase.

In [10]:
# ─────────────────────────────────────────────
# Plot Histograms for Each Iteration Count
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for idx, k in enumerate(iter_counts):
    counts = all_counts[k]
    ax     = axes[idx]

    top_states = sorted(counts.items(), key=lambda x: -x[1])[:20]
    states     = [s for s, _ in top_states]
    freqs      = [c for _, c in top_states]
    colors     = ['crimson' if s in targets else 'steelblue' for s in states]

    bars = ax.bar(states, freqs, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f"Grover — {k} Iteration(s)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Measured State", fontsize=10)
    ax.set_ylabel("Counts", fontsize=10)
    ax.tick_params(axis='x', rotation=90, labelsize=7)

    for bar, state in zip(bars, states):
        if state in targets:
            ax.annotate('★ MARKED',
                        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 6), textcoords='offset points',
                        ha='center', fontsize=7, color='crimson', fontweight='bold')

    legend = [mpatches.Patch(color='crimson',   label='Marked state'),
              mpatches.Patch(color='steelblue', label='Unmarked state')]
    ax.legend(handles=legend, fontsize=8)

plt.suptitle("Grover's Algorithm — 10-Qubit Search\nMarked: 0110011010 & 1101010001",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("grover_histograms.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Saved: grover_histograms.png")

Saved: grover_histograms.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\820870773.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ─────────────────────────────────────────────
# Summary Table
# ─────────────────────────────────────────────
print("=" * 60)
print(f"{'Iterations':<12} {'State':<15} {'Counts':<10} {'Probability'}")
print("=" * 60)
for k in iter_counts:
    counts = all_counts[k]
    total  = sum(counts.values())
    for t in targets:
        c    = counts.get(t, 0)
        prob = c / total * 100
        print(f"{k:<12} {t:<15} {c:<10} {prob:.2f}%")
    print("-" * 60)
print(f"\nOptimal iteration count (theory): {optimal_iters}")

Iterations   State           Counts     Probability
1            0110011010      14         1.37%
1            1101010001      14         1.37%
------------------------------------------------------------
3            0110011010      42         4.10%
3            1101010001      40         3.91%
------------------------------------------------------------
5            0110011010      119        11.62%
5            1101010001      106        10.35%
------------------------------------------------------------
10           0110011010      362        35.35%
10           1101010001      304        29.69%
------------------------------------------------------------

Optimal iteration count (theory): 18


---

## Conclusion

- The **Oracle** correctly identifies and phase-flips the two marked states using multi-controlled gates.
- The **Diffusion operator** amplifies marked state probabilities by inverting amplitudes about their mean.
- With increasing Grover iterations, the probability of measuring the marked states grows, peaking near the **optimal iteration count (~18)**.
- Beyond the optimal count, probability begins to decrease — a phenomenon called **over-rotation**.

### Key Results:
| Iterations | Expected behavior |
|------------|-------------------|
| 1          | Small boost visible |
| 3–5        | Marked states clearly dominant |
| 10         | High probability, near or past peak |
| ~18        | Maximum probability (theoretical optimum) |

The circuit was built **entirely from scratch** using basic Qiskit gates with no built-in Grover implementations.

In [12]:
# ─────────────────────────────────────────────
# Bonus: Run at Optimal Iteration Count (~18)
# ─────────────────────────────────────────────
print(f"Running at optimal iteration count: {optimal_iters}")
qc_opt     = build_grover_circuit(n_qubits, targets, optimal_iters)
compiled   = transpile(qc_opt, simulator)
job        = simulator.run(compiled, shots=2048)
result     = job.result()
counts_opt = result.get_counts()

print(f"\nTop 5 outcomes at {optimal_iters} iterations:")
for state, count in sorted(counts_opt.items(), key=lambda x: -x[1])[:5]:
    prob   = count / 2048 * 100
    marker = " ← MARKED ✓" if state in targets else ""
    print(f"  {state} : {count:5d} shots  ({prob:.1f}%){marker}")

# ── Optimal circuit diagram ───────────────────
oracle_gate_opt    = circuit_to_gate_clean(build_oracle(n_qubits, targets), label="Oracle")
diffusion_gate_opt = circuit_to_gate_clean(build_diffusion(n_qubits),       label="Diffusion")

qc_opt_view = QuantumCircuit(n_qubits, n_qubits)
qc_opt_view.h(range(n_qubits))
qc_opt_view.barrier()
for i in range(optimal_iters):
    qc_opt_view.append(oracle_gate_opt,    range(n_qubits))
    qc_opt_view.barrier()
    qc_opt_view.append(diffusion_gate_opt, range(n_qubits))
    qc_opt_view.barrier()
qc_opt_view.measure(range(n_qubits), range(n_qubits))

fig_opt = qc_opt_view.draw('mpl', style='clifford', fold=40)
fig_opt.set_size_inches(16, 5)
fig_opt.suptitle(f"Full Grover Circuit — Optimal ({optimal_iters}) Iterations\n"
                 f"H⊗¹⁰ → [Oracle → Diffusion]×{optimal_iters} → Measure",
                 fontsize=11, fontweight='bold')
fig_opt.savefig("grover_optimal_circuit.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig_opt)

# ── Optimal histogram ─────────────────────────
fig_hist = plot_histogram(counts_opt,
                          title=f"Grover at Optimal ({optimal_iters}) Iterations",
                          figsize=(14, 5))
fig_hist.savefig("grover_optimal_histogram.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig_hist)
print("Saved: grover_optimal_circuit.png, grover_optimal_histogram.png")

Running at optimal iteration count: 18

Top 5 outcomes at 18 iterations:
  1101010001 :  1046 shots  (51.1%) ← MARKED ✓
  0110011010 :   991 shots  (48.4%) ← MARKED ✓
  1111101100 :     1 shots  (0.0%)
  0000101110 :     1 shots  (0.0%)
  1100101000 :     1 shots  (0.0%)
Saved: grover_optimal_circuit.png, grover_optimal_histogram.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\1740677870.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9976\1740677870.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
